# LawReformer AI — Fine-Tuning Gemma 4 on 5000 Indian Legal Q&A Pairs

**Built by Rudrani Ghosh · [lawreformer.com](https://lawreformer.com) · © 2026 Rudrani Ghosh**

Kaggle Gemma 4 Good Hackathon · Digital Equity & Inclusivity Track

## What This Notebook Does
1. Loads 5000 Indian legal Q&A training pairs (English, Hindi, Bengali)
2. Fine-tunes Gemma-3-4b-it using Unsloth QLoRA (4-bit, fits T4 GPU)
3. Tests the fine-tuned model on legal questions
4. Exports model for deployment (HuggingFace + GGUF for edge)

## Dataset
- 5000 Q&A pairs covering 28 Indian laws
- Categories: rent control, wages, RTI, MGNREGA, domestic violence, consumer, legal aid, SC/ST, cyber crimes
- Languages: English (4909), Hindi (73), Bengali (18)
- Generated from 1395 legal provisions using template-based augmentation

In [ ]:
# Install dependencies
!pip install -q unsloth transformers datasets trl peft accelerate bitsandbytes huggingface_hub

In [ ]:
import json
from datasets import Dataset

# Load the 5000 training pairs
# Upload training_data.jsonl to Kaggle as input dataset
# Or place it in the notebook's working directory

import os
data_path = None
for path in ['training_data.jsonl', '/kaggle/input/lawreformer-training-data/training_data.jsonl', '../input/lawreformer-training-data/training_data.jsonl']:
    if os.path.exists(path):
        data_path = path
        break

if data_path is None:
    # Download from GitHub if not found locally
    !wget -q https://raw.githubusercontent.com/RudraniGhosh24/lawreform/main/notebooks/training_data.jsonl
    data_path = 'training_data.jsonl'

training_data = []
with open(data_path, 'r', encoding='utf-8') as f:
    for line in f:
        training_data.append(json.loads(line.strip()))

print(f'Loaded {len(training_data)} training pairs')
print(f'Sample: {training_data[0]["input"][:100]}...')
print(f'\nInstruction preview: {training_data[0]["instruction"][:80]}...')

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load Gemma 4B with 4-bit quantization (fits T4 16GB)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='google/gemma-3-4b-it',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Add LoRA adapters for efficient fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
)

print('Model loaded with LoRA adapters')
model.print_trainable_parameters()

In [ ]:
# Format dataset for Gemma chat template
def format_for_gemma(example):
    return {
        'text': f"<start_of_turn>user\n{example['instruction']}\n\n{example['input']}<end_of_turn>\n<start_of_turn>model\n{example['output']}<end_of_turn>"
    }

dataset = Dataset.from_list(training_data)
dataset = dataset.map(format_for_gemma)

print(f'Dataset size: {len(dataset)}')
print(f'\nFormatted example (first 300 chars):')
print(dataset[0]['text'][:300])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Fine-tune on 5000 pairs
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        output_dir='outputs',
        optim='adamw_8bit',
        seed=42,
        save_steps=500,
    ),
)

print('Starting fine-tuning on 5000 pairs...')
print('This will take ~2-3 hours on T4 GPU')
trainer_stats = trainer.train()
print(f'\nTraining complete!')
print(f'Final loss: {trainer_stats.training_loss:.4f}')
print(f'Total steps: {trainer_stats.global_step}')

In [ ]:
# Test the fine-tuned model
FastLanguageModel.for_inference(model)

test_questions = [
    'My landlord is not returning my security deposit. What are my rights?',
    'My employer has not paid my salary for 2 months. What can I do?',
    'How do I file an RTI application?',
    'I am facing domestic violence. Where can I get help?',
    'The gram panchayat is not giving me MGNREGA work. What are my rights?',
]

system = "You are LawReformer AI. Answer legal questions about Indian law in plain language. Cite specific laws."

for q in test_questions:
    prompt = f'<start_of_turn>user\n{system}\n\n{q}<end_of_turn>\n<start_of_turn>model\n'
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.7, do_sample=True)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response.split('<start_of_turn>model')[-1].strip()
    print(f'Q: {q}')
    print(f'A: {answer[:200]}...')
    print('-' * 80)

In [ ]:
# Test Hindi response
hindi_prompt = '<start_of_turn>user\nYou are LawReformer AI. Respond ENTIRELY in Hindi.\n\nमेरे मालिक ने दो महीने से वेतन नहीं दिया है। मैं क्या कर सकता हूँ?<end_of_turn>\n<start_of_turn>model\n'
inputs = tokenizer(hindi_prompt, return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.7, do_sample=True)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print('Hindi test:')
print(response.split('<start_of_turn>model')[-1].strip()[:300])

In [ ]:
# Save model locally
model.save_pretrained('lawreformer-gemma-4b-indian-legal')
tokenizer.save_pretrained('lawreformer-gemma-4b-indian-legal')
print('Model saved locally')

# Push to HuggingFace Hub (set your token)
# from huggingface_hub import login
# login(token='your_hf_token_here')
# model.push_to_hub('RudraniGhosh24/lawreformer-gemma-4b-indian-legal')
# tokenizer.push_to_hub('RudraniGhosh24/lawreformer-gemma-4b-indian-legal')
# print('Pushed to HuggingFace Hub')

In [ ]:
# Export as GGUF for edge deployment (Ollama/llama.cpp)
model.save_pretrained_gguf(
    'lawreformer-gemma-4b-gguf',
    tokenizer,
    quantization_method='q4_k_m'
)
print('GGUF model saved (q4_k_m quantization)')
print('This can run on: laptop (Ollama), Android (llama.cpp), Raspberry Pi 5')
print('\nTo run locally: ollama create lawreformer -f Modelfile')

## Results

### Training Stats
- **Model**: gemma-3-4b-it (Gemma 4 family)
- **Method**: QLoRA (4-bit quantization + LoRA r=16)
- **Dataset**: 5000 Indian legal Q&A pairs
- **Languages**: English, Hindi, Bengali
- **Laws covered**: 28 Indian statutes
- **Training time**: ~2-3 hours on T4 GPU
- **Framework**: Unsloth (2x faster than standard fine-tuning)

### Edge Deployment
The GGUF model (~2.5GB) runs on:
- **Laptop**: `ollama run lawreformer-gemma-4b`
- **Android**: via llama.cpp or MLC-LLM
- **Raspberry Pi 5**: with 8GB RAM
- **Browser**: via WebLLM (experimental)

This enables fully offline legal assistance — critical for rural India.

### Architecture
```
User (voice/text) → Web Speech API → RAG (1395 provisions) →
Fine-tuned Gemma 4 → Response with citations → TTS (voice output)
```

## Credits
Built by Rudrani Ghosh · [lawreformer.com](https://lawreformer.com)  
Kaggle Gemma 4 Good Hackathon 2026 · Digital Equity Track  
Fine-tuning: [Unsloth](https://github.com/unslothai/unsloth)  
Live demo: [ai.lawreformer.com](https://ai.lawreformer.com)